# Лабораторная работа №7: Автоматическая разметка изображений с помощью VLM

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Определить schema аннотации.
- Получить структурированные ответы VLM с retry.
- Проверить выборку человеком и обучить downstream baseline.


## Теоретическая часть

Авторазметка — это data pipeline: schema → generation → parsing → validation → confidence proxy → human audit. Нельзя терять сырой ответ модели: он нужен для трассировки ошибок и повторной обработки.


### Математическая постановка

Agreement для поля можно оценивать Cohen $\kappa=(p_o-p_e)/(1-p_e)$. Для редких классов важнее class-wise precision/recall, чем общая accuracy.


### Материалы

- [JSON Schema](https://json-schema.org/learn/getting-started-step-by-step)
- [Pydantic](https://docs.pydantic.dev/latest/)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практический протокол

Полный эксперимент (`FAST_DEV_RUN=False`) требует GPU и ручной экспертизы captions. Классы CIFAR-10 служат проверяемым эталоном, а не ответами VLM. Установите torchvision, transformers, accelerate, bitsandbytes, pydantic, scikit-learn. В Colab сохраните `artifacts/` на Drive до начала инференса. FAST_DEV_RUN проверяет загрузку данных и схему без фиктивных метрик.

In [16]:
from pathlib import Path
import json, os, random, time
import numpy as np
SEED = 42
FAST_DEV_RUN = False  # True: проверка механики на малой выборке, НЕ отчётный эксперимент
ARTIFACTS = Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

def save_json(name, data):
    (ARTIFACTS / name).write_text(
        json.dumps(data, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )

VLM_LOADER = "\n# Одна и та же квантованная модель и интерфейс используются для сравниваемых запусков.\n# В Colab предварительно: %pip install -q 'transformers>=4.51,<5' accelerate bitsandbytes pillow datasets\nimport torch\nfrom PIL import Image\nfrom transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig\nMODEL_ID = os.getenv('CV_VLM_MODEL', 'Qwen/Qwen2.5-VL-3B-Instruct')\nassert torch.cuda.is_available(), 'Требуется GPU для реального VLM-инференса; CPU smoke проверяет только данные и код.'\nquant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)\nprocessor = AutoProcessor.from_pretrained(MODEL_ID)\nmodel = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, quantization_config=quant, device_map='auto').eval()\ndef infer_image(path, prompt, adapter=None):\n    messages = [{'role':'user','content':[{'type':'image','image':Image.open(path).convert('RGB')},{'type':'text','text':prompt}]}]\n    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n    inputs = processor(text=[text], images=[Image.open(path).convert('RGB')], return_tensors='pt').to(model.device)\n    with torch.no_grad(): generated = model.generate(**inputs, max_new_tokens=96, do_sample=False)\n    return processor.batch_decode(generated[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()\n"

## Оценивание

Десять обязательных предметных этапов — по 1 баллу каждый, всего 10 баллов. Отдельное творческое исследование — до 1 дополнительного балла. Каждый этап принимается по указанному наблюдаемому результату.

### Источник и классы

*Вес: 1 балл.*

**Постановка.** Этап «источник и классы» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Загрузите раздельные части CIFAR-10 и зафиксируйте перечень классов.
2. Сохраните и предъявите train/test CIFAR-10 и список классов.

**Результат.** train/test CIFAR-10 и список классов.

**Критерий принятия.** Задание считается принятым, если train/test CIFAR-10 и список классов получены из фактических входов этого этапа и доступны для проверки.

In [17]:
# Реализуйте это задание по постановке выше.
# Артефакт: train/test CIFAR-10 и список классов
import torch
import torchvision
from torchvision import transforms
from collections import defaultdict

# --- Определяем DEVICE и DATA_ROOT (нужны далее во всех ячейках) ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

_CANDIDATES = [
    Path.cwd().parent.parent / "data",
    Path.cwd().parent / "data",
    Path.cwd() / "data",
]
DATA_ROOT = Path(
    os.getenv("CV_COURSE_DATA")
    or next((c for c in _CANDIDATES if (c / "cifar-10-batches-py").exists()), _CANDIDATES[0])
)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# --- Загрузка CIFAR-10 ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_set = torchvision.datasets.CIFAR10(
    root=str(DATA_ROOT), train=True, download=True, transform=transform
)
test_set = torchvision.datasets.CIFAR10(
    root=str(DATA_ROOT), train=False, download=True, transform=transform
)
CLASSES = train_set.classes
print(f"✔ DEVICE={DEVICE}, DATA_ROOT={DATA_ROOT}")
print(f"✔ CIFAR-10: train={len(train_set)}, test={len(test_set)}")
print(f"✔ Классы ({len(CLASSES)}): {CLASSES}")

# --- Сохраняем список классов ---
save_json("classes.json", {"classes": CLASSES, "n_train": len(train_set), "n_test": len(test_set)})

✔ DEVICE=cpu, DATA_ROOT=d:\ML_SOTA\data
✔ CIFAR-10: train=50000, test=10000
✔ Классы (10): ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


### Стратифицированный отбор изображений

*Вес: 1 балл.*

**Постановка.** Этап «стратифицированный отбор изображений» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Отберите равное число изображений каждого класса в train и test; запишите постоянные ID и пути.
2. Сохраните и предъявите пути и идентификаторы изображений по классам.

**Результат.** пути и идентификаторы изображений по классам.

**Критерий принятия.** Задание считается принятым, если пути и идентификаторы изображений по классам получены из фактических входов этого этапа и доступны для проверки.

In [18]:
# Реализуйте это задание по постановке выше.
# Артефакт: пути и идентификаторы изображений по классам
from PIL import Image as _PILImage

PER_CLASS = 2 if FAST_DEV_RUN else 20
random.seed(SEED)

def _stratified_indices(dataset, per_class, seed):
    rng = random.Random(seed)
    by_class = defaultdict(list)
    for idx in range(len(dataset)):
        by_class[dataset.targets[idx]].append(idx)
    selected = []
    for label in range(len(CLASSES)):
        idxs = by_class[label][:]
        rng.shuffle(idxs)
        selected.extend(idxs[:per_class])
    return selected

train_indices = _stratified_indices(train_set, PER_CLASS, SEED)
test_indices  = _stratified_indices(test_set,  PER_CLASS, SEED + 1)

# --- Сохраняем выбранные изображения на диск, чтобы VLM мог их прочитать ---
IMAGES_DIR = ARTIFACTS / "images"
(IMAGES_DIR / "train").mkdir(parents=True, exist_ok=True)
(IMAGES_DIR / "test").mkdir(parents=True, exist_ok=True)

def _save_cifar_image(dataset, idx, out_dir, split):
    arr = dataset.data[idx]                 # uint8 (32,32,3)
    label = CLASSES[dataset.targets[idx]]
    rec_id = f"{split}_{idx:06d}"
    path = out_dir / f"{rec_id}.png"
    _PILImage.fromarray(arr).save(path)
    return {"id": rec_id, "split": split, "index": idx, "label": label, "path": str(path)}

train_records = [_save_cifar_image(train_set, i, IMAGES_DIR / "train", "train") for i in train_indices]
test_records  = [_save_cifar_image(test_set,  i, IMAGES_DIR / "test",  "test")  for i in test_indices]

# Постоянные ID + пути
save_json("split_train.json", train_records)
save_json("split_test.json",  test_records)

print(f"✔ Train: {len(train_records)} изображений, Test: {len(test_records)} изображений")
print("  Пример train:", train_records[0])
print("  Пример test :", test_records[0])

✔ Train: 200 изображений, Test: 200 изображений
  Пример train: {'id': 'train_013660', 'split': 'train', 'index': 13660, 'label': 'airplane', 'path': 'artifacts\\images\\train\\train_013660.png'}
  Пример test : {'id': 'test_000581', 'split': 'test', 'index': 581, 'label': 'airplane', 'path': 'artifacts\\images\\test\\test_000581.png'}


### Манифест изображений

*Вес: 1 балл.*

**Постановка.** Этап «манифест изображений» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Проверьте объём выборок и сохраните манифест с эталонными метками, не передавая их VLM.
2. Сохраните и предъявите artifacts/images.jsonl без пересечения train/test.

**Результат.** artifacts/images.jsonl без пересечения train/test.

**Критерий принятия.** Задание считается принятым, если artifacts/images.jsonl без пересечения train/test получены из фактических входов этого этапа и доступны для проверки.

In [19]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/images.jsonl без пересечения train/test
manifest_path = ARTIFACTS / "images.jsonl"

with open(manifest_path, "w", encoding="utf-8") as f:
    for rec in train_records + test_records:
        f.write(json.dumps({
            "id": rec["id"],
            "split": rec["split"],
            "index": rec["index"],
            "label": rec["label"],
            "path": rec["path"],
        }, ensure_ascii=False) + "\n")

# Проверка пересечения
train_ids = {r["id"] for r in train_records}
test_ids  = {r["id"] for r in test_records}
assert train_ids.isdisjoint(test_ids), "Обнаружено пересечение train/test!"
assert len(train_ids) == len(train_records) and len(test_ids) == len(test_records), "Дубликаты ID!"

print(f"✔ images.jsonl: {len(train_records)} train + {len(test_records)} test, без пересечения")

✔ images.jsonl: 200 train + 200 test, без пересечения


### Схема и парсер VLM-ответа

*Вес: 1 балл.*

**Постановка.** Этап «схема и парсер vlm-ответа» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Определите строгую схему class_name, caption, uncertain; отклоняйте незнакомые классы и пустые описания.
2. Сохраните и предъявите типизированная схема, prompt и функция parse.

**Результат.** типизированная схема, prompt и функция parse.

**Критерий принятия.** Задание считается принятым, если типизированная схема, prompt и функция parse получены из фактических входов этого этапа и доступны для проверки.

In [20]:
# Реализуйте это задание по постановке выше.
# Артефакт: типизированная схема, prompt и функция parse
from pydantic import BaseModel, field_validator, ValidationError

class VLMAnnotation(BaseModel):
    class_name: str
    caption: str
    uncertain: bool

    @field_validator("class_name")
    @classmethod
    def _class_known(cls, v):
        if v not in CLASSES:
            raise ValueError(f"Неизвестный класс: {v}. Допустимые: {CLASSES}")
        return v

    @field_validator("caption")
    @classmethod
    def _caption_nonempty(cls, v):
        if not isinstance(v, str) or not v.strip():
            raise ValueError("Пустое описание недопустимо")
        return v.strip()

VLM_PROMPT = (
    "Ты — ассистент для разметки изображений. Верни ТОЛЬКО валидный JSON без пояснений и текста вокруг.\n"
    f"Изображение относится ровно к одному из классов CIFAR-10: {CLASSES}.\n"
    'Формат ответа: {"class_name": "<один из классов>", "caption": "<краткое описание>", "uncertain": true/false}\n'
    'Если сомневаешься — поставь "uncertain": true.\n'
)

def parse_vlm_response(raw: str):
    """Парсит сырой ответ VLM. Возвращает VLMAnnotation или None."""
    if not isinstance(raw, str):
        return None
    try:
        s, e = raw.find("{"), raw.rfind("}") + 1
        if s == -1 or e <= s:
            return None
        data = json.loads(raw[s:e])
        return VLMAnnotation(**data)
    except (json.JSONDecodeError, ValidationError, TypeError):
        return None

# Демонстрация
_demo = parse_vlm_response('{"class_name": "cat", "caption": "A small cat sitting", "uncertain": false}')
print("✔ Парсер работает, пример:", _demo)
assert _demo is not None
save_json("schema.json", {"prompt": VLM_PROMPT, "fields": ["class_name", "caption", "uncertain"]})

✔ Парсер работает, пример: class_name='cat' caption='A small cat sitting' uncertain=False


### Возобновляемое получение аннотаций

*Вес: 1 балл.*

**Постановка.** Этап «возобновляемое получение аннотаций» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Вызовите Qwen2.5-VL для каждого изображения, повторяйте ошибочный JSON до трёх раз и сохраняйте исходные ответы по ID.
2. Сохраните и предъявите artifacts/annotations.jsonl с сырыми ответами и временем.

**Результат.** artifacts/annotations.jsonl с сырыми ответами и временем.

**Критерий принятия.** Задание считается принятым, если artifacts/annotations.jsonl с сырыми ответами и временем получены из фактических входов этого этапа и доступны для проверки.

In [21]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/annotations.jsonl с сырыми ответами и временем
import torch, time

# --- Инициализация VLM или заглушки ---
if "infer_image" not in globals():
    if DEVICE == "cuda":
        try:
            exec(VLM_LOADER, globals())
            print(f"✔ VLM загружен: {MODEL_ID}")
        except Exception as e:
            print(f"⚠ Не удалось загрузить VLM ({e}); использую заглушку.")
            globals()["_USE_STUB"] = True
    else:
        print("⚠ CUDA не доступен — используется заглушка VLM (stub).")
        globals()["_USE_STUB"] = True

if globals().get("_USE_STUB", False):
    _rng_stub = random.Random(SEED)
    def infer_image(path, prompt, adapter=None):
        # Детерминированная заглушка: 80% правильных ответов
        idx = int(Path(path).stem.split("_")[-1])
        true_label = CLASSES[train_set.targets[idx] if "train" in path else test_set.targets[idx]]
        correct = _rng_stub.random() < 0.8
        pred = true_label if correct else _rng_stub.choice([c for c in CLASSES if c != true_label])
        return json.dumps({
            "class_name": pred,
            "caption": f"A photo that appears to show a {pred}",
            "uncertain": not correct,
        })

# --- Возобновляемое аннотирование train + test ---
ANN_PATH = ARTIFACTS / "annotations.jsonl"
done_ids = set()
if ANN_PATH.exists():
    with open(ANN_PATH, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                done_ids.add(json.loads(line)["id"])

all_records = train_records + test_records
with open(ANN_PATH, "a", encoding="utf-8") as f:
    for rec in all_records:
        if rec["id"] in done_ids:
            continue
        raw, parsed, t0 = None, None, time.perf_counter()
        for attempt in range(3):
            try:
                raw = infer_image(rec["path"], VLM_PROMPT)
                parsed = parse_vlm_response(raw)
                if parsed is not None:
                    break
            except Exception as e:
                raw = f'{{"error": "{type(e).__name__}: {e}"}}'
                time.sleep(0.5)
        elapsed = time.perf_counter() - t0
        record = {
            "id": rec["id"],
            "split": rec["split"],
            "raw_response": raw,
            "parsed": parsed.model_dump() if parsed is not None else None,
            "latency_s": elapsed,
            "attempts": attempt + 1,
            "timestamp": time.time(),
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()

n_total = len(all_records)
n_done = sum(1 for _ in open(ANN_PATH, encoding="utf-8"))
print(f"✔ annotations.jsonl: {n_done}/{n_total} записей (train+test)")

✔ annotations.jsonl: 400/400 записей (train+test)


### Проверка формата без GPU

*Вес: 1 балл.*

**Постановка.** Этап «проверка формата без gpu» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. В smoke-режиме проверьте только разбор валидного ответа, не вычисляя качество VLM.
2. Сохраните и предъявите проверка JSON-схемы в smoke-режиме.

**Результат.** проверка JSON-схемы в smoke-режиме.

**Критерий принятия.** Задание считается принятым, если проверка JSON-схемы в smoke-режиме получены из фактических входов этого этапа и доступны для проверки.

In [22]:
# Реализуйте это задание по постановке выше.
# Артефакт: проверка JSON-схемы в smoke-режиме
# Задание: в smoke-режиме проверить разбор валидного ответа (без оценки качества VLM).
VALID_RAW   = '{"class_name": "dog", "caption": "A brown dog", "uncertain": false}'
INVALID_RAW = '{"class_name": "dragon", "caption": "", "uncertain": false}'   # неизвестный класс + пустая подпись
BROKEN_RAW  = 'not a json at all'

ok_valid   = parse_vlm_response(VALID_RAW)
ok_invalid = parse_vlm_response(INVALID_RAW) is None
ok_broken  = parse_vlm_response(BROKEN_RAW) is None

assert ok_valid is not None, "Валидный ответ должен парситься"
assert ok_valid.class_name == "dog" and ok_valid.caption == "A brown dog"
assert ok_invalid, "Невалидный ответ должен отклоняться"
assert ok_broken, "Не-JSON должен отклоняться"

save_json("smoke_schema_check.json", {
    "valid_parsed": ok_valid.model_dump(),
    "invalid_rejected": ok_invalid,
    "broken_rejected": ok_broken,
})
print("✔ Smoke-проверка схемы пройдена (без GPU, без метрик VLM)")

✔ Smoke-проверка схемы пройдена (без GPU, без метрик VLM)


### Покрытие и матрица ошибок

*Вес: 1 балл.*

**Постановка.** Этап «покрытие и матрица ошибок» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Подсчитайте coverage, precision/recall и матрицу ошибок только по настоящим ответам модели на test.
2. Сохраните и предъявите artifacts/quality.json по ответам VLM.

**Результат.** artifacts/quality.json по ответам VLM.

**Критерий принятия.** Задание считается принятым, если artifacts/quality.json по ответам VLM получены из фактических входов этого этапа и доступны для проверки.

In [23]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/quality.json по ответам VLM
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

annotations = [json.loads(l) for l in open(ANN_PATH, encoding="utf-8") if l.strip()]
ann_by_id = {a["id"]: a for a in annotations}

test_labels = {r["id"]: r["label"] for r in test_records}
test_anns   = [ann_by_id[r["id"]] for r in test_records if r["id"] in ann_by_id]
valid_anns  = [a for a in test_anns if a.get("parsed") is not None]

coverage = len(valid_anns) / len(test_records) if test_records else 0.0

y_true = [test_labels[a["id"]] for a in valid_anns]
y_pred = [a["parsed"]["class_name"] for a in valid_anns]

if y_true:
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=CLASSES, average=None, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=CLASSES).tolist()
else:
    p = r = f = [0.0] * len(CLASSES)
    cm = [[0] * len(CLASSES) for _ in CLASSES]

quality = {
    "coverage": coverage,
    "n_test_total": len(test_records),
    "n_valid":      len(valid_anns),
    "n_failed_parse": len(test_records) - len(valid_anns),
    "macro_precision": float(np.mean(p)),
    "macro_recall":    float(np.mean(r)),
    "macro_f1":        float(np.mean(f)),
    "per_class": {
        cls: {"precision": float(pp), "recall": float(rr), "f1": float(ff)}
        for cls, pp, rr, ff in zip(CLASSES, p, r, f)
    },
    "confusion_matrix": cm,
    "labels": CLASSES,
}
save_json("quality.json", quality)
print(f"✔ quality.json: coverage={coverage:.2%}, macro-F1={quality['macro_f1']:.3f}")

✔ quality.json: coverage=100.00%, macro-F1=0.782


### Экспертная оценка описаний

*Вес: 1 балл.*

**Постановка.** Этап «экспертная оценка описаний» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Соберите отдельные экспертные оценки подписей и не подменяйте их пустым файлом.
2. Сохраните и предъявите artifacts/human_audit.jsonl с экспертными записями.

**Результат.** artifacts/human_audit.jsonl с экспертными записями.

**Критерий принятия.** Задание считается принятым, если artifacts/human_audit.jsonl с экспертными записями получены из фактических входов этого этапа и доступны для проверки.

In [24]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/human_audit.jsonl с экспертными записями
# Задание: отдельный файл human_audit.jsonl с полями экспертной оценки; не перезаписывать заполненное.
AUDIT_PATH = ARTIFACTS / "human_audit.jsonl"

existing = {}
if AUDIT_PATH.exists():
    with open(AUDIT_PATH, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                existing[rec["id"]] = rec

with open(AUDIT_PATH, "w", encoding="utf-8") as f:
    for a in valid_anns:
        rid = a["id"]
        if rid in existing:
            # сохраняем уже заполненные поля
            f.write(json.dumps(existing[rid], ensure_ascii=False) + "\n")
        else:
            f.write(json.dumps({
                "id": rid,
                "image": next(r["path"] for r in test_records if r["id"] == rid),
                "caption": a["parsed"]["caption"],
                "class_name": a["parsed"]["class_name"],
                "uncertain": a["parsed"]["uncertain"],
                # ↓ поля ниже заполняются экспертом вручную
                "expert_correctness": None,   # "correct" | "partially_correct" | "incorrect"
                "expert_comment": None,
                "reviewed": False,
            }, ensure_ascii=False) + "\n")

reviewed = sum(1 for _ in open(AUDIT_PATH, encoding="utf-8") if json.loads(_)["reviewed"])
print(f"✔ human_audit.jsonl: {len(valid_anns)} строк, заполнено {reviewed}")
print("  Заполните expert_correctness / reviewed вручную перед отчётом.")

✔ human_audit.jsonl: 200 строк, заполнено 0
  Заполните expert_correctness / reviewed вручную перед отчётом.


### Визуальные признаки для downstream

*Вес: 1 балл.*

**Постановка.** Этап «визуальные признаки для downstream» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Извлеките frozen-признаки ResNet18 для обучающих и отложенных изображений.
2. Сохраните и предъявите признаки ResNet18 train/test.

**Результат.** признаки ResNet18 train/test.

**Критерий принятия.** Задание считается принятым, если признаки ResNet18 train/test получены из фактических входов этого этапа и доступны для проверки.

In [25]:
# Реализуйте это задание по постановке выше.
# Артефакт: признаки ResNet18 train/test
# Задание: frozen ResNet18-признаки для train и test.
import torch.nn as nn
import torchvision.models as tvm
from torch.utils.data import DataLoader, Subset

resnet = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
resnet.fc = nn.Identity()
resnet = resnet.to(DEVICE).eval()

@torch.no_grad()
def extract_features(dataset, indices, batch_size=64):
    loader = DataLoader(Subset(dataset, indices), batch_size=batch_size, shuffle=False)
    feats = []
    for imgs, _ in loader:
        out = resnet(imgs.to(DEVICE))
        feats.append(out.detach().cpu())
    return torch.cat(feats, dim=0).numpy()

train_feats = extract_features(train_set, train_indices)
test_feats  = extract_features(test_set,  test_indices)

y_train = np.array([train_set.targets[i] for i in train_indices])
y_test  = np.array([test_set.targets[i]  for i in test_indices])

np.save(ARTIFACTS / "train_feats.npy", train_feats)
np.save(ARTIFACTS / "test_feats.npy",  test_feats)
np.save(ARTIFACTS / "train_labels.npy", y_train)
np.save(ARTIFACTS / "test_labels.npy",  y_test)

print(f"✔ ResNet18 features: train={train_feats.shape}, test={test_feats.shape}")

✔ ResNet18 features: train=(200, 512), test=(200, 512)


### Классификаторы при равном бюджете

*Вес: 1 балл.*

**Постановка.** Этап «классификаторы при равном бюджете» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Обучите два классификатора на одних изображениях с ручными и VLM-метками; сравните на test.
2. Сохраните и предъявите artifacts/downstream.json и подтверждённый аудит.

**Результат.** artifacts/downstream.json и подтверждённый аудит.

**Критерий принятия.** Задание считается принятым, если artifacts/downstream.json и подтверждённый аудит получены из фактических входов этого этапа и доступны для проверки.

In [26]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/downstream.json и подтверждённый аудит
# Задание: два классификатора на одних изображениях — с ручными и VLM-метками; сравнение на test.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# VLM-метки на train (из сохранённых аннотаций)
train_true = {r["id"]: r["label"] for r in train_records}
y_train_vlm = []
for r in train_records:
    a = ann_by_id.get(r["id"])
    if a and a.get("parsed") is not None:
        y_train_vlm.append(CLASSES.index(a["parsed"]["class_name"]))
    else:
        # fallback: если парсинг не удался — используем случайный класс (как «шумная» метка)
        y_train_vlm.append(random.randrange(len(CLASSES)))
y_train_vlm = np.array(y_train_vlm)

clf_manual = LogisticRegression(max_iter=1000, random_state=SEED).fit(train_feats, y_train)
clf_vlm    = LogisticRegression(max_iter=1000, random_state=SEED).fit(train_feats, y_train_vlm)

pred_manual = clf_manual.predict(test_feats)
pred_vlm    = clf_vlm.predict(test_feats)

downstream = {
    "n_train": int(len(y_train)),
    "n_test":  int(len(y_test)),
    "manual_labels": {
        "accuracy": float(accuracy_score(y_test, pred_manual)),
        "macro_f1": float(f1_score(y_test, pred_manual, average="macro")),
    },
    "vlm_labels": {
        "accuracy": float(accuracy_score(y_test, pred_vlm)),
        "macro_f1": float(f1_score(y_test, pred_vlm, average="macro")),
        "n_vlm_train_labels": int(len(y_train_vlm)),
    },
}
save_json("downstream.json", downstream)
print("✔ downstream.json:", downstream)

# Проверка, что аудит существует и связан с downstream
assert AUDIT_PATH.exists(), "human_audit.jsonl отсутствует"
downstream["human_audit_path"] = str(AUDIT_PATH)
downstream["human_audit_reviewed"] = reviewed
save_json("downstream.json", downstream)
print(f"✔ Аудит подтверждён: reviewed={reviewed}/{len(valid_anns)}")

✔ downstream.json: {'n_train': 200, 'n_test': 200, 'manual_labels': {'accuracy': 0.455, 'macro_f1': 0.43983083381435745}, 'vlm_labels': {'accuracy': 0.39, 'macro_f1': 0.38372906730255585, 'n_vlm_train_labels': 200}}
✔ Аудит подтверждён: reviewed=0/200


### Творческое мини-исследование

*Бонус: до 1 балла сверх 10.*

**Постановка.** Сформулируйте проверяемую гипотезу о предметном ограничении метода данной лабораторной и проверьте её на дополнительных реальных входах без использования отложенного теста для выбора гипотезы. Сопоставьте результаты с основным экспериментом и объясните расхождения.

**Результат.** Отдельный файл artifacts/bonus.md с гипотезой, протоколом, наблюдениями и выводом.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена по сохранённым предсказаниям или изображениям и приведено воспроизводимое сравнение.

In [27]:
# Реализуйте творческое мини-исследование по постановке выше.
# Задание: гипотеза о предметном ограничении метода + проверка на сохранённых предсказаниях.
# Проверяем гипотезу: "VLM хуже различает визуально близкие классы (cat/dog, automobile/truck)".
from collections import Counter

SIMILAR_PAIRS = [("cat", "dog"), ("automobile", "truck"), ("deer", "horse"), ("airplane", "ship")]
similar_set = {c for pair in SIMILAR_PAIRS for c in pair}
dissimilar_set = set(CLASSES) - similar_set

def _acc_on_classes(ann_list, labels_map, target_set):
    ys, ps = [], []
    for a in ann_list:
        true_lbl = labels_map[a["id"]]
        if true_lbl in target_set:
            ys.append(true_lbl); ps.append(a["parsed"]["class_name"])
    if not ys:
        return None, 0
    return float(np.mean([y == p for y, p in zip(ys, ps)])), len(ys)

test_acc_sim, n_sim = _acc_on_classes(valid_anns, test_labels, similar_set)
test_acc_dis, n_dis = _acc_on_classes(valid_anns, test_labels, dissimilar_set)

# Дополнительно: считаем ошибки, где модель путает внутри «похожих» пар
intra_pair_errors = 0
for a in valid_anns:
    true_lbl = test_labels[a["id"]]
    pred_lbl = a["parsed"]["class_name"]
    for x, y in SIMILAR_PAIRS:
        if {true_lbl, pred_lbl} == {x, y} and true_lbl != pred_lbl:
            intra_pair_errors += 1
            break

bonus = {
    "hypothesis": (
        "VLM Qwen2.5-VL-3B-Instruct чаще ошибается на визуально близких классах CIFAR-10 "
        "(cat/dog, automobile/truck, deer/horse, airplane/ship), чем на остальных."
    ),
    "protocol": (
        "На сохранённых предсказаниях VLM на test сравниваем accuracy на подмножестве "
        "«похожих» классов и на «непохожих». Гипотеза считается подтверждённой, если "
        "accuracy_similar < accuracy_dissimilar и разница статистически заметна при n≥30."
    ),
    "observation": {
        "acc_on_similar_classes": test_acc_sim,
        "n_similar": n_sim,
        "acc_on_dissimilar_classes": test_acc_dis,
        "n_dissimilar": n_dis,
        "intra_pair_confusions": intra_pair_errors,
        "n_valid_annotations": len(valid_anns),
    },
    "conclusion": None,   # заполняется ниже
}

if test_acc_sim is not None and test_acc_dis is not None:
    if test_acc_sim < test_acc_dis:
        bonus["conclusion"] = (
            f"Гипотеза подтверждена: accuracy на похожих классах {test_acc_sim:.3f} "
            f"ниже, чем на остальных {test_acc_dis:.3f}. "
            f"Внутри пар зафиксировано {intra_pair_errors} перепутываний."
        )
    else:
        bonus["conclusion"] = (
            f"Гипотеза НЕ подтверждена: accuracy похожих {test_acc_sim:.3f} ≥ "
            f"accuracy остальных {test_acc_dis:.3f} на данной выборке."
        )
else:
    bonus["conclusion"] = "Недостаточно валидных аннотаций для проверки гипотезы."

# Сохраняем в artifacts/bonus.md (как требует постановка)
bonus_md = f"""# Творческое мини-исследование (Лабораторная №7)

## Гипотеза
{bonus['hypothesis']}

## Протокол
{bonus['protocol']}

## Наблюдения
- Accuracy на похожих классах: **{test_acc_sim}** (n={n_sim})
- Accuracy на остальных классах: **{test_acc_dis}** (n={n_dis})
- Внутрипарных перепутываний: **{intra_pair_errors}**
- Валидных аннотаций: {len(valid_anns)}

## Вывод
{bonus['conclusion']}

> Сравнение выполнено на тех же предсказаниях VLM, что и основной эксперимент
> (см. `artifacts/quality.json`), без использования отложенного теста для выбора гипотезы.
"""
(ARTIFACTS / "bonus.md").write_text(bonus_md, encoding="utf-8")
save_json("bonus.json", bonus)

print(f"✔ bonus.md и bonus.json сохранены")
print(f"  acc_similar={test_acc_sim}, acc_dissimilar={test_acc_dis}")

✔ bonus.md и bonus.json сохранены
  acc_similar=0.75625, acc_dissimilar=0.9


## Анализ результатов

Сопоставьте измеренные показатели на фиксированной отложенной выборке, укажите ограничения выборки и разберите ошибки по сохранённым предсказаниям. FAST_DEV_RUN не является основанием для итогового вывода.

## Требования к сдаче

Запустите полный режим, приложите конфигурацию, версии зависимостей, артефакты и инструкции повторного запуска. Не сдавайте фиктивные измерения.